# Can we predict late delivery?
Target: `late_delivery`. Prediction is made at dispatch, for consignments later than those we have data for.

Decisions up front (checklist.md):
- **Split**: rows are time-ordered and the model is for future rows, so the test set is the latest 20% by `dispatched_on`. Each `consignment` appears once, so there is no entity grouping to respect.
- **CV**: expanding-window `TimeSeriesSplit` inside the training rows only.
- **Leakage**: `claim_paid_eur` is recorded after delivery (a claim settled for the consignment), so it is not known at dispatch. It is dropped, as is the identifier `consignment`.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.consignment.is_unique, df.isna().sum().sum())
df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.agg(['mean','size'])

The monthly late rate is far from stable (about 30% in 2024, below 5% from Oct 2025). The latest rows are therefore unlike earlier ones, so a temporal test is a hard, honest check. Calendar month/trend is not used as a feature, since it cannot extrapolate to future periods.

In [ ]:
df['hour'] = df.dispatched_on.dt.hour
df['dow'] = df.dispatched_on.dt.dayofweek
FEATURES = ['carrier','distance_km','weight_kg','parcels','hour','dow']
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
print('late rate train %.3f test %.3f' % (train.late_delivery.mean(), test.late_delivery.mean()))
X_tr, y_tr = train[FEATURES], train.late_delivery
# test set is not touched again until the final cell

## Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: one-hot carrier + scaled numerics, logistic regression (C=1).
3. `logreg_C0.1`: same with stronger regularisation (C=0.1).
4. `hgb`: HistGradientBoosting (max_depth=3, lr=0.05, 200 iter).

Selection metric: CV mean ROC AUC. The decision threshold is then chosen on out-of-fold predictions to maximise F1. All preprocessing sits inside the pipelines, so it is fit per fold.

In [ ]:
num = ['distance_km','weight_kg','parcels','hour','dow']
def pre(): return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier']), ('n', StandardScaler(), num)])
models = {
 'baseline': DummyClassifier(strategy='prior'),
 'logreg': make_pipeline(pre(), LogisticRegression(C=1, max_iter=1000)),
 'logreg_C0.1': make_pipeline(pre(), LogisticRegression(C=0.1, max_iter=1000)),
 'hgb': make_pipeline(ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier'])], remainder='passthrough'),
                      HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
tscv = TimeSeriesSplit(n_splits=5)
rows, oof = [], {}
for name, m in models.items():
    p = np.full(len(train), np.nan); aucs=[]; aps=[]
    for tr, va in tscv.split(X_tr):
        f = clone(m).fit(X_tr.iloc[tr], y_tr.iloc[tr])
        p[va] = f.predict_proba(X_tr.iloc[va])[:,1]
        aucs.append(roc_auc_score(y_tr.iloc[va], p[va])); aps.append(average_precision_score(y_tr.iloc[va], p[va]))
    oof[name] = p
    rows.append(dict(model=name, cv_auc=np.mean(aucs), auc_sd=np.std(aucs), cv_ap=np.mean(aps)))
cv = pd.DataFrame(rows).set_index('model'); cv

In [ ]:
best = cv.drop('baseline').cv_auc.idxmax(); print('chosen:', best)
mask = ~np.isnan(oof[best])
ths = np.linspace(0.05, 0.8, 76)
f1s = [f1_score(y_tr[mask], oof[best][mask] >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('threshold %.2f, CV F1 %.3f' % (thr, max(f1s)))
print('predict-all-late CV F1:', round(f1_score(y_tr[mask], np.ones(mask.sum())),3))

## Decision recorded before scoring the test set
Model, features and threshold are those printed above, chosen on CV only: the non-trivial candidate with the highest CV ROC AUC, features `carrier, distance_km, weight_kg, parcels, hour, dow`, and the F1-maximising out-of-fold threshold. Nothing is changed after seeing the test score.

In [ ]:
final = clone(models[best]).fit(X_tr, y_tr)
base = clone(models['baseline']).fit(X_tr, y_tr)
p = final.predict_proba(test[FEATURES])[:,1]; pb = base.predict_proba(test[FEATURES])[:,1]
yt = test.late_delivery
print(pd.Series({
 'model': best, 'threshold': thr,
 'test AUC': roc_auc_score(yt, p), 'test AP': average_precision_score(yt, p), 'test prevalence': yt.mean(),
 'test F1 @thr': f1_score(yt, p>=thr), 'test Brier': brier_score_loss(yt, p),
 'baseline Brier': brier_score_loss(yt, pb)}))

## Conclusion
Headline (single test-set score, latest 20% of rows, `logreg_C0.1`): ROC AUC 0.70, versus CV AUC 0.72. So the dispatch-time features (carrier, distance, weight, parcels, hour, weekday) rank late consignments modestly better than chance.

Caveats:
- The late rate collapsed from 25.7% in training to 2.2% in the test period. Ranking transfers (AUC), but calibration and thresholds do not: at the CV-chosen threshold F1 is 0.08, and the Brier score (0.088) is worse than the training-prior baseline (0.077). As deployed today the model would over-flag lateness.
- Whatever drove the drop in late rate is not in the features. Recalibrate on recent data before any operational use.
- `claim_paid_eur` almost perfectly separates late from on-time (it exists after the outcome), so it was excluded as leakage.